In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import xgboost as xgb
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import plot_importance

sys.path.append("..")

from challenge.data.etl import transform_data

warnings.filterwarnings("ignore")

## 0. Load Data

In [ ]:
data = pd.read_csv("../data/data.csv")
data.info()

## 4. Training

### 4.a. Data Split (Training and Validation)

In [ ]:
features, target = transform_data(data, target_column="delay")

In [ ]:
features.head()

In [ ]:
target.head()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.33, random_state=42)

In [ ]:
print(f"train shape: {x_train.shape} | test shape: {x_test.shape}")

In [ ]:
y_train.value_counts("%") * 100

In [ ]:
y_test.value_counts("%") * 100

### 4.b. Model Selection

Setup MLFlow autologging and tracking URI.

In [ ]:
mlflow.autolog()
mlflow.set_tracking_uri("http://localhost:5500")

Generate Eval Data for MLFlow

In [ ]:
eval_data = x_test.copy()
eval_data["label"] = y_test

#### 4.b.i. XGBoost

In [ ]:
xgb_model = xgb.XGBClassifier(random_state=1, learning_rate=0.01)
with mlflow.start_run(run_name="XGBoost Classifier"):
    xgb_model.fit(x_train, y_train)
    signature = infer_signature(x_test, xgb_model.predict(x_test))
    model_info = mlflow.xgboost.log_model(xgb_model, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets="label",
        model_type="classifier",
    )

In [ ]:
xgboost_y_preds = xgb_model.predict(x_test)
xgboost_y_preds = [1 if y_pred > 0.5 else 0 for y_pred in xgboost_y_preds]

In [ ]:
confusion_matrix(y_test, xgboost_y_preds)

In [ ]:
print(classification_report(y_test, xgboost_y_preds))

#### 4.b.ii. Logistic Regression

In [ ]:
reg_model = LogisticRegression()
with mlflow.start_run(run_name="Logistic Regression Classifier"):
    reg_model.fit(x_train, y_train)
    signature = infer_signature(x_test, reg_model.predict(x_test))
    model_info = mlflow.sklearn.log_model(reg_model, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets="label",
        model_type="classifier",
    )

In [ ]:
reg_y_preds = reg_model.predict(x_test)

In [ ]:
confusion_matrix(y_test, reg_y_preds)

In [ ]:
print(classification_report(y_test, reg_y_preds))

## 5. Data Analysis: Third Sight

### Feature Importance

In [ ]:
plt.figure(figsize=(10, 5))
plot_importance(xgb_model)

In [ ]:
top_10_features = [
    "OPERA_Latin American Wings",
    "MES_7",
    "MES_10",
    "OPERA_Grupo LATAM",
    "MES_12",
    "TIPOVUELO_I",
    "MES_4",
    "MES_11",
    "OPERA_Sky Airline",
    "OPERA_Copa Air",
]

### Data Balance

In [ ]:
n_y0 = len(y_train[y_train == 0])
n_y1 = len(y_train[y_train == 1])
scale = n_y0 / n_y1
print(scale)

## 6. Training with Improvement

### 6.a. Data Split

In [ ]:
x_train2, x_test2, y_train2, y_test2 = train_test_split(
    features[top_10_features], target, test_size=0.33, random_state=42
)

### 6.b. Model Selection

Generate Eval Data for MLFlow

In [ ]:
eval_data2 = x_test2.copy()
eval_data2["label"] = y_test2

#### 6.b.i. XGBoost with Feature Importance and with Balance

In [ ]:
xgb_model_2 = xgb.XGBClassifier(random_state=1, learning_rate=0.01, scale_pos_weight=scale)
with mlflow.start_run(run_name="XGBoost Classifier - Top 10 Features"):
    xgb_model_2.fit(x_train2, y_train2)
    signature = infer_signature(x_test2, xgb_model_2.predict(x_test2))
    model_info = mlflow.xgboost.log_model(xgb_model_2, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data2,
        targets="label",
        model_type="classifier",
    )

In [ ]:
xgboost_y_preds_2 = xgb_model_2.predict(x_test2)

In [ ]:
confusion_matrix(y_test2, xgboost_y_preds_2)

In [ ]:
print(classification_report(y_test2, xgboost_y_preds_2))

#### 6.b.ii. XGBoost with Feature Importance but without Balance

In [ ]:
xgb_model_3 = xgb.XGBClassifier(random_state=1, learning_rate=0.01)
with mlflow.start_run(run_name="XGBoost Classifier - Top 10 Features - No Scale Pos Weight"):
    xgb_model_3.fit(x_train2, y_train2)
    signature = infer_signature(x_test2, xgb_model_3.predict(x_test2))
    model_info = mlflow.xgboost.log_model(xgb_model_3, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data2,
        targets="label",
        model_type="classifier",
    )

In [ ]:
xgboost_y_preds_3 = xgb_model_3.predict(x_test2)

In [ ]:
confusion_matrix(y_test2, xgboost_y_preds_3)

In [ ]:
print(classification_report(y_test2, xgboost_y_preds_3))

#### 6.b.iii. Logistic Regression with Feature Importante and with Balance

In [ ]:
reg_model_2 = LogisticRegression(class_weight={1: n_y0 / len(y_train), 0: n_y1 / len(y_train)})
with mlflow.start_run(run_name="Logistic Regression Classifier - Top 10 Features - Class Weight"):
    reg_model_2.fit(x_train2, y_train2)
    signature = infer_signature(x_test2, reg_model_2.predict(x_test2))
    model_info = mlflow.sklearn.log_model(reg_model_2, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data2,
        targets="label",
        model_type="classifier",
    )

In [ ]:
reg_y_preds_2 = reg_model_2.predict(x_test2)

In [ ]:
confusion_matrix(y_test2, reg_y_preds_2)

In [ ]:
print(classification_report(y_test2, reg_y_preds_2))

#### 6.b.iv. Logistic Regression with Feature Importante but without Balance

In [ ]:
reg_model_3 = LogisticRegression()
with mlflow.start_run(run_name="Logistic Regression Classifier - Top 10 Features - No Class Weight"):
    reg_model_3.fit(x_train2, y_train2)
    signature = infer_signature(x_test2, reg_model_3.predict(x_test2))
    model_info = mlflow.sklearn.log_model(reg_model_3, name="model", signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data2,
        targets="label",
        model_type="classifier",
    )

In [ ]:
reg_y_preds_3 = reg_model_3.predict(x_test2)

In [ ]:
confusion_matrix(y_test2, reg_y_preds_3)

In [ ]:
print(classification_report(y_test2, reg_y_preds_3))

## 7. Data Science Conclusions

By looking at the results of the 6 trained models, it can be determined:
- There is no noticeable difference in results between XGBoost and LogisticRegression.
- Does not decrease the performance of the model by reducing the features to the 10 most important.
- Improves the model's performance when balancing classes, since it increases the recall of class "1".

**With this, the model to be productive must be the one that is trained with the top 10 features and class balancing, but which one?**